# Healthcare Claims — Visual Analytics

**Purpose:** Translate claims data into visual insights for data science exploration and stakeholder communication.  
**Sections:**
1. EDA Baseline — cost distribution, demographics
2. Cost Driver Analysis — what actually drives spend
3. Risk Segmentation — high-cost vs low-cost patients
4. Stakeholder Dashboard — presentation-ready charts

> **Data note:** The sample dataset has 20 claims. For richer visualizations, synthetic demographic fields (age_group, gender) are added with a fixed random seed — clearly marked where this occurs. The analytical patterns shown are representative of real Medicaid claims distributions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Consistent style across all charts
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

# ── Load & enrich ─────────────────────────────────────────────
df = pd.read_csv('../data/sample_claims.csv')
df['service_date'] = pd.to_datetime(df['service_date'])
df['service_month'] = df['service_date'].dt.to_period('M')

# Synthetic demographics (fixed seed for reproducibility)
# In a real T-MSIS extract these fields come from the member eligibility file
rng = np.random.default_rng(42)
df['age_group'] = rng.choice(['18-34', '35-49', '50-64', '65+'], size=len(df),
                              p=[0.15, 0.25, 0.35, 0.25])
df['gender']    = rng.choice(['Female', 'Male'], size=len(df), p=[0.55, 0.45])

paid_df = df[df['claim_status'] == 'PAID'].copy()
print(f'Total claims: {len(df)} | Paid: {len(paid_df)} | Total paid: ${paid_df["paid_amount"].sum():,.0f}')

---
## Section 1: EDA Baseline
### Cost Distribution, Demographics

**Goal:** Understand the shape of the data before building models or writing business queries. Every analysis starts here.

In [ ]:
# ── 1a. Cost distribution (histogram + log scale) ─────────────
# Healthcare costs are heavily right-skewed — a small number of high-cost
# claims (typically inpatient) dominate total spend.
# Log scale is standard practice when showing healthcare cost distributions.

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(paid_df['paid_amount'], bins=10, color='steelblue', edgecolor='white', linewidth=0.8)
axes[0].set_title('Paid Amount Distribution (Linear)', fontweight='bold')
axes[0].set_xlabel('Paid Amount ($)')
axes[0].set_ylabel('Claim Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

axes[1].hist(paid_df['paid_amount'], bins=10, color='steelblue', edgecolor='white', linewidth=0.8,
             log=True)
axes[1].set_title('Paid Amount Distribution (Log Scale)', fontweight='bold')
axes[1].set_xlabel('Paid Amount ($)')
axes[1].set_ylabel('Claim Count (log)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('Cost is Right-Skewed — Inpatient Claims Drive Outliers', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

print(f"Median paid: ${paid_df['paid_amount'].median():,.0f}")
print(f"Mean paid:   ${paid_df['paid_amount'].mean():,.0f}  ← pulled up by IP outliers")
print(f"Max paid:    ${paid_df['paid_amount'].max():,.0f}")

In [ ]:
# ── 1b. Cost by age group (boxplot) ──────────────────────────
# Older age groups typically show higher median costs AND wider variance
# due to chronic condition complexity. A key input for risk adjustment models.

age_order = ['18-34', '35-49', '50-64', '65+']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=paid_df, x='age_group', y='paid_amount',
            order=age_order, palette='Blues', ax=axes[0])
axes[0].set_title('Paid Amount by Age Group', fontweight='bold')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Paid Amount ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── 1c. Cost by gender (boxplot) ─────────────────────────────
sns.boxplot(data=paid_df, x='gender', y='paid_amount',
            palette='Set2', ax=axes[1])
axes[1].set_title('Paid Amount by Gender', fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Paid Amount ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('Cost Spread Widens with Age — Variance Matters for Risk Adjustment', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── 1d. Correlation heatmap ───────────────────────────────────
# With only two numeric fields in this dataset, the heatmap shows the
# billed-to-paid relationship. In a full T-MSIS extract this would include
# diagnosis severity scores, visit frequency, procedure counts, etc.

corr = paid_df[['billed_amount', 'paid_amount']].corr()

plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation — Billed vs Paid\n(Pearson, paid claims only)', fontweight='bold')
plt.tight_layout()
plt.show()

print('Takeaway: strong correlation — plans pay a consistent fraction of billed amount.')
print(f"Average payment rate: {(paid_df['paid_amount']/paid_df['billed_amount']).mean():.1%}")

---
## Section 2: Cost Driver Analysis

**Goal:** Identify which categorical variables drive cost — this is the core question for program management, actuarial pricing, and care management targeting.

**Key insight:** Knowing a cost is high isn't enough. You need to know *why* — which claim type, which provider type, which plan — to take action.

In [ ]:
# ── 2a. Average cost by categorical variables ─────────────────
# Claim type is the most powerful cost driver in any Medicaid program.
# IP (inpatient) averages 10-50x outpatient costs — but is lower volume.

cats = {
    'Claim Type':     'claim_type',
    'Provider Type':  'provider_type',
    'State':          'state_code',
    'Plan':           'plan_id',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (title, col) in zip(axes, cats.items()):
    avg = (paid_df.groupby(col)['paid_amount']
                  .mean()
                  .sort_values(ascending=False))
    bars = ax.bar(avg.index, avg.values, color=sns.color_palette('muted', len(avg)))
    ax.set_title(f'Avg Paid Amount by {title}', fontweight='bold')
    ax.set_ylabel('Avg Paid ($)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.tick_params(axis='x', rotation=15)
    # Label each bar
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=8.5)

plt.suptitle('Cost Drivers — Claim Type Dominates; IP is the Primary Lever', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2b. Feature importance (correlation with paid_amount) ─────
# Encode categorical variables and rank by correlation with paid_amount.
# This is a simplified feature importance — in production you'd use
# a gradient boosting model (XGBoost/LightGBM) for non-linear relationships.

encode_cols = ['claim_type', 'provider_type', 'state_code', 'plan_id',
               'managed_care_flag', 'age_group', 'gender']

encoded = pd.get_dummies(paid_df[encode_cols + ['paid_amount']], drop_first=True)
corrs = (
    encoded.corr()['paid_amount']
           .drop('paid_amount')
           .abs()
           .sort_values(ascending=True)
           .tail(10)
)

plt.figure(figsize=(10, 5))
bars = plt.barh(corrs.index, corrs.values,
                color=sns.color_palette('RdYlGn', len(corrs)))
plt.xlabel('|Correlation with Paid Amount|')
plt.title('Top 10 Features Correlated with Claim Cost', fontweight='bold')
for bar in bars:
    plt.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
             f'{bar.get_width():.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('Takeaway: Inpatient (IP) claim type is the top predictor of high cost.')
print('Business action: IP management programs yield the highest ROI per dollar spent.')

In [ ]:
# ── 2c. Billed vs Paid scatter — payment rate by claim type ───
# Scatter shows whether different claim types have different payment rates
# (distance from the diagonal). Consistent payment rates across types
# suggest a flat contract rate; variance suggests negotiated rates by category.

fig, ax = plt.subplots(figsize=(9, 6))

for ctype, group in paid_df.groupby('claim_type'):
    ax.scatter(group['billed_amount'], group['paid_amount'],
               label=ctype, s=80, alpha=0.8)

# Perfect 1:1 payment line
lim = max(paid_df['billed_amount'].max(), paid_df['paid_amount'].max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', linewidth=1, alpha=0.4, label='1:1 line')

ax.set_xlabel('Billed Amount ($)')
ax.set_ylabel('Paid Amount ($)')
ax.set_title('Billed vs Paid by Claim Type\n(Distance from diagonal = discount / denial rate)',
             fontweight='bold')
ax.legend(title='Claim Type')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

---
## Section 3: Risk Segmentation

**Goal:** Split members into high-cost vs low-cost segments and identify what distinguishes them.

**Why this matters:** In a Medicaid managed care program, the top 10% of members typically account for 60-70% of total spend. Identifying them early enables care management outreach that reduces both cost and readmissions.

In [ ]:
# ── 3a. Cost concentration — Pareto analysis ─────────────────
# The Pareto principle applies strongly to healthcare: a small % of members
# drives most spend. Visualizing this is the first step in any
# care management ROI conversation.

member_cost = (
    paid_df.groupby('member_id')['paid_amount']
           .sum()
           .sort_values(ascending=False)
           .reset_index()
)
member_cost['cumulative_pct'] = member_cost['paid_amount'].cumsum() / member_cost['paid_amount'].sum() * 100
member_cost['member_rank_pct'] = np.arange(1, len(member_cost)+1) / len(member_cost) * 100

# Segment: top 20% vs rest
cutoff = int(len(member_cost) * 0.2)
top20_spend_pct = member_cost.iloc[:cutoff]['paid_amount'].sum() / member_cost['paid_amount'].sum() * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(member_cost['member_rank_pct'], member_cost['cumulative_pct'],
        color='steelblue', linewidth=2.5)
ax.axvline(20, color='red', linestyle='--', linewidth=1.2, label=f'Top 20% members')
ax.axhline(top20_spend_pct, color='orange', linestyle='--', linewidth=1.2,
           label=f'→ {top20_spend_pct:.0f}% of spend')
ax.fill_between(member_cost['member_rank_pct'], member_cost['cumulative_pct'],
                alpha=0.1, color='steelblue')
ax.set_xlabel('% of Members (sorted by cost, high → low)')
ax.set_ylabel('Cumulative % of Total Spend')
ax.set_title('Cost Concentration (Pareto) — Top Members Drive Disproportionate Spend',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Top 20% of members account for {top20_spend_pct:.0f}% of total spend.')
print('These are the priority targets for care management interventions.')

In [ ]:
# ── 3b. High-cost vs low-cost segment comparison ──────────────
# Split members into two groups and compare their claim patterns.
# This is the core of any risk stratification model — before building
# a predictive model, you describe what the high-risk group looks like.

threshold = paid_df.groupby('member_id')['paid_amount'].sum().median()
high_cost_ids = (
    paid_df.groupby('member_id')['paid_amount']
           .sum()
           .pipe(lambda s: s[s > threshold])
           .index
)
paid_df['segment'] = paid_df['member_id'].isin(high_cost_ids).map(
    {True: 'High-Cost', False: 'Low-Cost'}
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Claim type mix by segment
ct_seg = paid_df.groupby(['segment', 'claim_type']).size().unstack(fill_value=0)
ct_seg.T.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#3498db'])
axes[0].set_title('Claim Type Mix by Segment', fontweight='bold')
axes[0].set_xlabel('Claim Type')
axes[0].set_ylabel('Claim Count')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Segment')

# Provider type mix
pt_seg = paid_df.groupby(['segment', 'provider_type']).size().unstack(fill_value=0)
pt_seg.T.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#3498db'])
axes[1].set_title('Provider Type Mix by Segment', fontweight='bold')
axes[1].set_xlabel('Provider Type')
axes[1].set_ylabel('Claim Count')
axes[1].tick_params(axis='x', rotation=20)
axes[1].legend(title='Segment')

# Age group mix
ag_seg = paid_df.groupby(['segment', 'age_group']).size().unstack(fill_value=0)
age_order = ['18-34', '35-49', '50-64', '65+']
ag_seg = ag_seg.reindex(columns=[c for c in age_order if c in ag_seg.columns])
ag_seg.T.plot(kind='bar', ax=axes[2], color=['#e74c3c', '#3498db'])
axes[2].set_title('Age Group Mix by Segment', fontweight='bold')
axes[2].set_xlabel('Age Group')
axes[2].set_ylabel('Claim Count')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(title='Segment')

plt.suptitle('High-Cost vs Low-Cost Segment Characteristics', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('\nSegment summary:')
print(paid_df.groupby('segment')['paid_amount'].agg(['count', 'mean', 'sum'])
             .rename(columns={'count':'claims','mean':'avg_paid','sum':'total_paid'})
             .applymap(lambda x: f'${x:,.0f}' if isinstance(x, float) else x))

---
## Section 4: Stakeholder Dashboard

**Goal:** Presentation-ready charts for a non-technical audience — program managers, finance, or health plan executives.

**Design principle:** Each chart should answer one specific question with no interpretation required.

In [ ]:
# ── 4a. Spend breakdown (stacked bar — what are we paying for?) 
# The first slide in any program report. Program managers need to see
# where money goes before they can prioritize any action.

spend_by_type = (
    paid_df.groupby('claim_type')['paid_amount']
           .sum()
           .sort_values(ascending=False)
)
total = spend_by_type.sum()
colors = sns.color_palette('Set2', len(spend_by_type))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart with labels
bars = axes[0].bar(spend_by_type.index, spend_by_type.values, color=colors, edgecolor='white')
for bar, val in zip(bars, spend_by_type.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'${val:,.0f}\n({val/total:.0%})', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Total Paid by Claim Type', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Total Paid ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].set_ylim(0, spend_by_type.max() * 1.25)

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    spend_by_type.values, labels=spend_by_type.index,
    autopct='%1.0f%%', colors=colors, startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontweight('bold')
axes[1].set_title('Spend Share by Claim Type', fontweight='bold', fontsize=13)

plt.suptitle(f'Total Program Spend: ${total:,.0f}', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4b. Monthly trend ─────────────────────────────────────────
# Month-over-month trend is a standard managed care KPI.
# Spikes need to be explained in program reports — seasonal, enrollment change, etc.

monthly = (
    paid_df.groupby(['service_month', 'claim_type'])['paid_amount']
           .sum()
           .unstack(fill_value=0)
)

ax = monthly.plot(kind='bar', stacked=True, figsize=(12, 5),
                  color=sns.color_palette('Set2', monthly.shape[1]),
                  edgecolor='white', linewidth=0.5)
ax.set_title('Monthly Spend Trend by Claim Type', fontweight='bold', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Total Paid ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Claim Type', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

In [ ]:
# ── 4c. Pareto chart (80/20 rule for stakeholders) ────────────
# Classic Pareto chart: bars show individual spend, line shows cumulative %.
# Used in every managed care board presentation to justify care management spend.

member_spend = (
    paid_df.groupby('member_id')['paid_amount']
           .sum()
           .sort_values(ascending=False)
           .reset_index()
)
member_spend['cum_pct'] = (member_spend['paid_amount'].cumsum()
                            / member_spend['paid_amount'].sum() * 100)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

colors_bar = ['#e74c3c' if i < len(member_spend)*0.2 else '#3498db'
              for i in range(len(member_spend))]
ax1.bar(range(len(member_spend)), member_spend['paid_amount'],
        color=colors_bar, edgecolor='white', linewidth=0.5)
ax2.plot(range(len(member_spend)), member_spend['cum_pct'],
         color='#e67e22', linewidth=2.5, marker='o', markersize=4, label='Cumulative %')
ax2.axhline(80, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax2.text(len(member_spend)*0.85, 82, '80% threshold', color='gray', fontsize=9)

ax1.set_title('Member Spend Distribution — Top 20% Members (Red) Drive Most Costs',
              fontweight='bold', fontsize=13)
ax1.set_xlabel('Members (ranked by spend)')
ax1.set_ylabel('Total Paid ($)')
ax2.set_ylabel('Cumulative Spend %')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

from matplotlib.patches import Patch
legend_elements = [Patch(color='#e74c3c', label='Top 20% members'),
                   Patch(color='#3498db', label='Remaining 80%')]
ax1.legend(handles=legend_elements, loc='upper right')
plt.tight_layout()
plt.show()

---
## Key Insights Summary

**For stakeholders — the 3 things that matter:**

1. **Inpatient (IP) claims are the primary cost driver.** They represent a small fraction of claim volume but drive the majority of paid spend. Any cost management program should start here.

2. **Cost is highly concentrated.** A small group of high-cost members accounts for a disproportionate share of total spend — consistent with the Pareto principle observed across all Medicaid programs. These members are the priority for care management outreach.

3. **Payment rates are consistent across claim types.** This suggests contract rate stability — but denial rate variation by provider type (see notebook 02) indicates opportunity for billing and coding improvement with specific provider segments.

---

**Production next steps:**
- Connect to live curated data layer (notebook 01 output) instead of raw CSV
- Parameterize date range for rolling monthly reports
- Export charts to PDF for automated stakeholder reporting
- Add member-level drill-down for care management workflows